In [17]:
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)

import requests
import urllib3  # Para capturar erros de protocolo
from dateutil.relativedelta import relativedelta
from datetime import datetime
import pandas as pd
import numpy as np
import zipfile
from datetime import date
import matplotlib.pyplot as plt
from io import StringIO, BytesIO
from bs4 import BeautifulSoup
import shutil
import os
import re
import time
import easygui

In [18]:
df = pd.read_parquet(fr"..\data\download_gov\anp_parquet_og.parquet") #anp_dolar_enriched.parquet")
df.head(2)

,Regiao - Sigla,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Nome da Rua,Numero Rua,Complemento,Bairro,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira,Regiao - Sigla.1
0,NE,CE,SOBRAL,AUTO POSTO APRAZIVEL LTDA,00.422.849/0001-53,RODOVIA BR 222,S/N,KM 248,SANTA CRUZ DO BANABUIU,00000-000,DIESEL,2004-05-10,1.470,1.507225,R$ / litro,PETROBRAS DISTRIBUIDORA S.A.,None
1,NE,BA,JAGUAQUARA,LUZITALIA COM. E TRANSP. DE DERIVADOS DE PETRO...,01.073.545/0001-90,AV PRESIDENTE MEDICI N. 150 BR 116 KM640,S/N,GALPAO,ENTRONCAMENTO,00000-000,DIESEL,2004-05-10,1.229,1.153090,R$ / litro,COSAN LUBRIFICANTES,None


In [19]:
df["Data da Coleta"].min()

'2004-05-10'

In [20]:
base_path = (
    rf"..\data\download_gov"
    # "c:\Users\ferna\Dev\Projetos\CondaProjects\ANP - Project ML_EDA\data\download_gov"
)
path_csv = os.path.join(base_path, "csv")
path_zip = os.path.join(base_path, "zip")
path_temp = os.path.join(base_path, "temp")  # Path definido para temp


In [21]:
def validate_and_convert_date_updated(date_str):
    """Valida e converte datas de 'dd/mm/aa' para 'dd/mm/yyyy'."""
    pattern_two_digit_year = r"^\d{2}/\d{2}/\d{2}$"
    pattern_four_digit_year = r"^\d{2}/\d{2}/\d{4}$"

    if re.match(pattern_two_digit_year, date_str):
        day, month, year = date_str.split("/")
        year = "20" + year
        return "/".join([day, month, year])

    elif re.match(pattern_four_digit_year, date_str):
        return date_str

    else:
        return float("nan")
    
def processar_dataframe(df):
    """
    Limpa, formata e processa o DataFrame de entrada.
    Esta é a etapa mais importante para garantir a concatenação correta.
    """

    # Limpa nomes das colunas ANTES de qualquer outra coisa
    df.columns = df.columns.str.replace("ï»¿", "")  # Remove BOM
    df.columns = (
        df.columns.str.normalize("NFKD")
        .str.encode("ascii", errors="ignore")
        .str.decode("ascii")
    )  # Remove acentos
    df.columns = df.columns.str.strip()  # Remove espaços extras

    # Processamento de Datas
    if "Data da Coleta" in df.columns:
        df["Data da Coleta"] = df["Data da Coleta"].astype(str)
        df["Data da Coleta"] = df["Data da Coleta"].apply(
            validate_and_convert_date_updated
        )
        df = df.dropna(subset=["Data da Coleta"])
        df["Data da Coleta"] = pd.to_datetime(df["Data da Coleta"],format="%d/%m/%Y")#
        df = df.sort_values(by=["CNPJ da Revenda", "Data da Coleta"])

    # Processamento de Valores (Venda)
    if "Valor de Venda" in df.columns:
        df["Valor de Venda"] = df["Valor de Venda"].astype(str)
        df["Valor de Venda"] = (
            df["Valor de Venda"]
            .str.replace(".", "", regex=False)
            .str.replace(",", ".")
            .astype(float)
        )
        df["Valor de Venda"].interpolate(method="linear", inplace=True)

    # Processamento de Valores (Compra)
    if "Valor de Compra" in df.columns:
        df["Valor de Compra"] = df["Valor de Compra"].astype(str)
        df["Valor de Compra"] = (
            df["Valor de Compra"]
            .str.replace(".", "", regex=False)
            .str.replace(",", ".")
            .astype(float)
        )
        df["Valor de Compra"].interpolate(method="linear", inplace=True)

    return df


In [22]:
df_og = pd.read_parquet(fr"..\data\download_gov\anp_parquet_og.parquet")

In [23]:
df_og.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22497213 entries, 0 to 22497212
Data columns (total 17 columns):
 #   Column             Dtype  
---  ------             -----  
 0   Regiao - Sigla     object 
 1   Estado - Sigla     object 
 2   Municipio          object 
 3   Revenda            object 
 4   CNPJ da Revenda    object 
 5   Nome da Rua        object 
 6   Numero Rua         object 
 7   Complemento        object 
 8   Bairro             object 
 9   Cep                object 
 10  Produto            object 
 11  Data da Coleta     object 
 12  Valor de Venda     float64
 13  Valor de Compra    float64
 14  Unidade de Medida  object 
 15  Bandeira           object 
 16  Regiao - Sigla.1   object 
dtypes: float64(2), object(15)
memory usage: 2.8+ GB


In [24]:
df_og["Data da Coleta"] = pd.to_datetime(df_og["Data da Coleta"], format="%Y-%m-%d")
df_og = df_og.drop(columns=["Regiao - Sigla.1"])

In [25]:

dfs = []

# Loop através de todos os arquivos na pasta CSV
for filename in os.listdir(path_csv):
    if filename.endswith(".csv"):
        print(f"Processando CSV: {filename}")
        file_path = os.path.join(path_csv, filename)
        try:
            df = pd.read_csv(file_path, sep=";", encoding="latin1", low_memory=False)
            df_processado = processar_dataframe(df)  # Usa a função refatorada
            dfs.append(df_processado)
        except Exception as e:
            print(f"Erro ao processar {filename}: {e}")

# Loop através de todos os arquivos na pasta ZIP
for filename in os.listdir(path_zip):
    if filename.endswith(".zip"):
        print(f"Processando ZIP: {filename}")
        file_path = os.path.join(path_zip, filename)

        try:
            with zipfile.ZipFile(file_path, "r") as z:
                # Lógica robusta para achar o CSV dentro do ZIP
                csv_filename = None
                for name in z.namelist():
                    if name.endswith(".csv"):
                        csv_filename = name
                        break  # Pega o primeiro CSV que encontrar

                if csv_filename:
                    with z.open(csv_filename) as f:
                        df = pd.read_csv(
                            f, sep=";", encoding="latin1", low_memory=False
                        )
                        df_processado = processar_dataframe(
                            df
                        )  # Usa a função refatorada
                        dfs.append(df_processado)
                else:
                    print(f"Nenhum arquivo .csv encontrado em {filename}")
        except Exception as e:
            print(f"Erro ao processar {filename}: {e}")

print("Processamento concluído. Concatenando DataFrames...")

# --- 6. Finalização ---
if dfs:  # Garante que a lista não está vazia
    df_extract = pd.concat(dfs, ignore_index=True)

    # Passos finais de limpeza
    df_extract = df_extract.drop_duplicates()
    df_extract = df_extract.sort_values(
        by=["Data da Coleta", "Produto", "Cep", "CNPJ da Revenda"]
    )


    # concatenando df extraido com o ultimo arquivo .parquet(df_og)
    final_df = pd.concat([df_og, df_extract], ignore_index=True)
    print("Salvando arquivo final anp.parquet...")
    final_df.to_parquet(f"{base_path}\\anp.parquet", index=False)
    print("Processo finalizado!")
    final_df.info()
else:
    print("Nenhum dado foi processado. Verifique os downloads e os caminhos.")

Processando CSV: 01-dados-abertos-precos-diesel-gnv.csv
Processando CSV: 01-dados-abertos-precos-gasolina-etanol.csv
Processando CSV: 01-dados-abertos-precos-glp.csv
Processando CSV: 02-cados-abertos-preco-gasolina-etanol.csv
Processando CSV: 02-dados-abertos-precos-diesel-gnv.csv
Processando CSV: 02-dados-abertos-precos-glp.csv
Processando CSV: precos-diesel-gnv-07.csv
Processando CSV: precos-diesel-gnv-08.csv
Processando CSV: precos-diesel-gnv-09.csv
Processando CSV: precos-diesel-gnv-10.csv
Processando CSV: precos-diesel-gnv-11.csv
Processando CSV: precos-diesel-gnv-12.csv
Processando CSV: precos-gasolina-etanol-07.csv
Processando CSV: precos-gasolina-etanol-08.csv
Processando CSV: precos-gasolina-etanol-09.csv
Processando CSV: precos-gasolina-etanol-10.csv
Processando CSV: precos-gasolina-etanol-11.csv
Processando CSV: precos-gasolina-etanol-12.csv
Processando CSV: precos-glp-07.csv
Processando CSV: precos-glp-08.csv
Processando CSV: precos-glp-09.csv
Processando CSV: precos-glp-10

C:\Users\ferna\AppData\Local\Temp\ipykernel_38628\762569405.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Data da Coleta"] = pd.to_datetime(df["Data da Coleta"],format="%d/%m/%Y")#


Processamento concluído. Concatenando DataFrames...
Salvando arquivo final anp.parquet...
Processo finalizado!
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23588880 entries, 0 to 23588879
Data columns (total 16 columns):
 #   Column             Dtype         
---  ------             -----         
 0   Regiao - Sigla     object        
 1   Estado - Sigla     object        
 2   Municipio          object        
 3   Revenda            object        
 4   CNPJ da Revenda    object        
 5   Nome da Rua        object        
 6   Numero Rua         object        
 7   Complemento        object        
 8   Bairro             object        
 9   Cep                object        
 10  Produto            object        
 11  Data da Coleta     datetime64[ns]
 12  Valor de Venda     float64       
 13  Valor de Compra    float64       
 14  Unidade de Medida  object        
 15  Bandeira           object        
dtypes: datetime64[ns](1), float64(2), object(13)
memory usage: 2.8+ GB


In [26]:
df = pd.read_parquet(r"..\data\download_gov\anp.parquet")


In [29]:
df["Data da Coleta"].max()

Timestamp('2026-03-27 00:00:00')